# jevmark: frozen-base baseline B0 (task 1.6)

Thin wrapper: clone the private repo at one commit, install pinned dependencies, build the data, and run `scripts/evaluate.py --ckpt base` for Qwen3-0.6B-Base and Qwen3-1.7B-Base. All logic lives in the repo. Setup and download steps are in `docs/KAGGLE.md`.

Settings: accelerator GPU T4 x2, internet on, secret `GITHUB_TOKEN` attached (a fine-grained, read-only token for this one repository).

In [ ]:
# Parameters: set before running. COMMIT must be a full 40-character sha.
REPO = "OWNER/jevmark"
COMMIT = "0000000000000000000000000000000000000000"
LIMIT = 30  # records per split in the smoke run
SIZES = ["06b", "17b"]  # backbone sizes to evaluate, in order; for example ["17b"] for session B of the 1.7B plan


In [ ]:
# Clone REPO at COMMIT. The token reaches git only through environment variables,
# is never put on a command line or in .git/config, and is redacted from any output.
import base64
import os
import re
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

assert re.fullmatch(r"[\w.-]+/[\w.-]+", REPO), "REPO must be owner/name"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "COMMIT must be a full 40-character sha"
assert SIZES and set(SIZES) <= {"06b", "17b"}, "SIZES must be a non-empty list of 06b and 17b"
# Let the CUDA caching allocator grow segments instead of fragmenting (decision 45). PyTorch 2.9 and later
# read PYTORCH_ALLOC_CONF; earlier releases read only PYTORCH_CUDA_ALLOC_CONF, so both are set. Every later
# !python inherits them.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path("/tmp/jevmark")  # outside /kaggle/working, so the notebook output holds only runs/


def run(cmd, env=None, secrets=()):
    result = subprocess.run(cmd, cwd=WORK, env=env, capture_output=True, text=True)
    output = result.stdout + result.stderr
    for secret in secrets:
        output = output.replace(secret, "***")
    if output.strip():
        print(output[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"{cmd[0]} {cmd[1]} failed with exit code {result.returncode}")


token = UserSecretsClient().get_secret("GITHUB_TOKEN")
header = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": header,
}
WORK.mkdir(parents=True, exist_ok=True)
if not (WORK / ".git").exists():
    run(["git", "init", "-q"])
    run(["git", "remote", "add", "origin", f"https://github.com/{REPO}.git"])
run(["git", "fetch", "-q", "--depth", "1", "origin", COMMIT], env=git_env, secrets=(token, header))
run(["git", "checkout", "-q", "--force", "FETCH_HEAD"])
del token, header, git_env

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=WORK, capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, f"checked out {head}, expected {COMMIT}"
os.chdir(WORK)
print("checked out", head)

# Committed run directories are history, not state (decision 45): a fresh clone holds earlier runs, which must never
# pass for this session's result. Delete every training run directory, and the B0 runs this session will write.
import shutil

stale = sorted(WORK.glob("runs/sft_*")) + sorted(WORK.glob("runs/fast_*"))
stale += [WORK / "runs" / name for size in SIZES for name in (f"base_{size}", f"base_{size}_limit{LIMIT}")]
for path in stale:
    shutil.rmtree(path, ignore_errors=True)
print("removed committed run directories:", ", ".join(p.name for p in stale))


In [ ]:
# The Hugging Face packages pinned to uv.lock; Kaggle keeps its own torch and numpy (decision 23). jevmark itself without deps.
# Kaggle's preinstalled torchao 0.10 makes peft 0.21 raise on LoRA adapter injection; jevmark does not use it.
!pip uninstall -y -q torchao
!pip install -q -r requirements-kaggle.txt
!pip install -q -e . --no-deps
!python -c "import sys, torch, transformers, peft, datasets; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'transformers', transformers.__version__, 'peft', peft.__version__, 'datasets', datasets.__version__)"

In [ ]:
# Build the eight JSONL splits (about half a minute); the build fails loudly if any check fails.
!make data-build PY=python

In [ ]:
# Smoke run first: LIMIT records per split (a seeded stratified sample) on the first size in SIZES.
# Writes runs/base_{size}_limit{LIMIT}/ (gitignored).
!python scripts/evaluate.py --ckpt base --config configs/base_{SIZES[0]}.yaml --limit {LIMIT} --device cuda
if _exit_code != 0:
    raise RuntimeError(f"smoke evaluation exited with code {_exit_code}")


In [ ]:
# B0, the frozen base, on all nine splits in full for every size in SIZES (decision 46): writes runs/base_{size}/.
for size in SIZES:
    !python scripts/evaluate.py --ckpt base --config configs/base_{size}.yaml --device cuda
    if _exit_code != 0:
        raise RuntimeError(f"base_{size} evaluation exited with code {_exit_code}")


In [ ]:
# Copy runs/ to /kaggle/working/runs for download, and show what was produced.
import json
import shutil

shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)
# Check only the runs this session evaluated; other committed runs in the clone come from other commits.
for name in [f"base_{size}" for size in SIZES]:
    metrics_path = Path("/kaggle/working/runs") / name / "metrics.json"
    metrics = json.loads(metrics_path.read_text())
    print(name, "commit", metrics["git"]["commit"], "dirty", metrics["git"]["dirty"],
          "fp32 fallback", metrics["precision"]["fp32_fallback_used"], f"{metrics['wall_clock_seconds'] / 60:.1f} min")
    assert metrics["git"]["commit"] == COMMIT, name